# Live CUDA in the notebook with `nvcc4jupyter`
Each `%%cuda` cell is a real, editable nvcc program. Edit, Shift-Enter, see it run.

In [1]:
%load_ext nvcc4jupyter
from nvcc4jupyter import set_defaults
set_defaults(compiler_args='-arch=sm_89 -O3 -std=c++17')

Source files will be saved in "/tmp/tmpjg6z48hn".


## A real demo as a live cell — 32 lanes sort 32 values via `__shfl` (edit me)

In [2]:
%%cuda
#include <cstdio>
#include <vector>
#include <random>
#include <algorithm>
__device__ int warp_oddeven_sort(int v){
  int lane = threadIdx.x & 31;
  for (int i=0;i<32;++i){
    int phase=i&1; bool is_low=((lane&1)==phase);
    int partner=is_low?lane+1:lane-1; bool active=(unsigned)partner<32u;
    int pv=__shfl_sync(0xffffffffu,v,active?partner:lane);
    int lo=min(v,pv),hi=max(v,pv); v=active?(is_low?lo:hi):v;
  }
  return v;
}
__global__ void warp_sort(int* g){ int t=blockIdx.x*blockDim.x+threadIdx.x; g[t]=warp_oddeven_sort(g[t]); }
int main(){
  const int n=1<<20; std::vector<int> h(n); std::mt19937 rng(1); for(auto&x:h)x=rng();
  int*d; cudaMalloc(&d,n*sizeof(int)); cudaMemcpy(d,h.data(),n*sizeof(int),cudaMemcpyHostToDevice);
  warp_sort<<<n/256,256>>>(d); cudaDeviceSynchronize();
  std::vector<int> out(n); cudaMemcpy(out.data(),d,n*sizeof(int),cudaMemcpyDeviceToHost);
  bool ok=true; for(int w=0;w<n;w+=32){ std::vector<int> r(h.begin()+w,h.begin()+w+32);
    std::sort(r.begin(),r.end()); for(int i=0;i<32;i++) if(out[w+i]!=r[i]) ok=false; }
  printf("32 lanes sort 32 values via __shfl only: [%s]\n", ok?"PASS":"FAIL");
}

32 lanes sort 32 values via __shfl only: [PASS]



## Inline profiling (`-p`) — coalesced vs strided, the sectors/request symptom
`%%cuda -p` runs the program under Nsight Compute; here we pull load sectors/request for both kernels.

In [3]:
%%cuda -p --profiler-args "--metrics l1tex__average_t_sectors_per_request_pipe_lsu_mem_global_op_ld.ratio --launch-count 2 --kernel-name regex:copy"
#include <cstdio>
__global__ void copy_coalesced(const int* in,int* out,int n){ int i=blockIdx.x*blockDim.x+threadIdx.x; if(i<n) out[i]=in[i]; }
__global__ void copy_strided  (const int* in,int* out,int n){ int i=blockIdx.x*blockDim.x+threadIdx.x; int j=(i*16)%n; if(i<n) out[j]=in[j]; }
int main(){ int n=1<<22; size_t b=(size_t)n*sizeof(int); int*in,*out; cudaMalloc(&in,b); cudaMalloc(&out,b);
  copy_coalesced<<<n/256,256>>>(in,out,n); copy_strided<<<n/256,256>>>(in,out,n); cudaDeviceSynchronize(); }

==PROF== Connected to process 795213 (/tmp/tmpjg6z48hn/9ef1e29f-c691-44e1-8b6b-00eb106325e8/cuda_exec.out)
==PROF== Profiling "copy_coalesced": 0%....50%....100% - 3 passes
==PROF== Profiling "copy_strided": 0%....50%....100% - 3 passes
==PROF== Disconnected from process 795213
[795213] cuda_exec.out@127.0.0.1
  copy_coalesced(const int *, int *, int) (16384, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 8.9
    Section: Command line profiler metrics
    -------------------------------------------------------------------- ----------- ------------
    Metric Name                                                          Metric Unit Metric Value
    -------------------------------------------------------------------- ----------- ------------
    l1tex__average_t_sectors_per_request_pipe_lsu_mem_global_op_ld.ratio      sector            4
    -------------------------------------------------------------------- ----------- ------------

  copy_strided(const int *, int *, int) (16384,